# 40_dl: LSTM forecast of next-cycle SOH

**Owner:** Khải Hưng · **Review:** Minh Vũ · **Plan:** `docs/Go.md` GĐ5d, WBS C6.7–C6.10, decisions in `docs/Rec.md` 6.1.

Code lives in `src/lstm_model.py`; this notebook runs it step by step and shows the results.

## Step 1. Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))

from protocol import load_split
from lstm_model import STEP_FEATURES, WINDOW, check_windows, make_windows

## Step 2. Shared data split

`load_split()` (shared by all four models) keeps tier-A rows without `cap_outlier` / `break_in` and splits by battery. `is_eval` marks the last 40% of each test battery, the rows every model predicts.

In [2]:
train, test = load_split()
assert (len(train), len(test), test.is_eval.sum()) == (947, 528, 213)
len(train), len(test), test.is_eval.sum()

(947, 528, np.int64(213))

## Step 3. Sliding windows (C6.7)

The target cycle *t* is predicted from the 10 cycles *t−9..t*. Each step holds what is known just before that cycle's discharge, the same information set as the SVR features:

| Step feature | Meaning |
|---|---|
| `soh_prev`, `T_max_prev_C` | result of the previous discharge |
| `chg_cc_time_s`, `Re_ohm`, `Rct_ohm`, `log_rest_h` | measured before this discharge |
| `is_4C`, `I_level_A`, `cutoff_V` | fixed operating conditions |

Rules: windows are built inside one battery; windows with missing Re/Rct (before the first EIS of B0005–B0007) are dropped, not filled; the network learns ΔSOH = SOH_t − SOH_{t−1}.

In [3]:
Xtr, dtr, ptr, itr = make_windows(train)
Xte, dte, pte, ite = make_windows(test)
check_windows(train, Xtr, dtr, ptr, itr)   # one battery per window, no same-cycle leakage
check_windows(test, Xte, dte, pte, ite)
assert len(Xtr) == 867 - 2 * 18           # 18 windows each of B0005, B0006 touch missing Re/Rct
assert set(test.index[test.is_eval]) <= set(ite), "every evaluated row needs a window"
print("train", Xtr.shape, " test", Xte.shape)
pd.DataFrame({"train": train.loc[itr].battery_id.value_counts(),
              "test": test.loc[ite].battery_id.value_counts()}).fillna(0).astype(int)

train (831, 10, 9)  test (470, 10, 9)


,train,test
battery_id,,
B0005,139,0
B0006,138,0
B0007,0,139
B0018,122,0
B0033,168,0
B0034,0,184
B0045,57,0
B0046,58,0
B0047,0,58


One window as the network sees it (first training window of B0018): rows are steps *t−9..t*, the target is the ΔSOH of the last row's cycle.

In [4]:
k = np.flatnonzero(train.loc[itr].battery_id.to_numpy() == "B0018")[0]
w = pd.DataFrame(Xtr[k], columns=STEP_FEATURES)
w.insert(0, "cycle_valid", train.cycle_valid.loc[itr[k] - WINDOW + 1:itr[k]].to_numpy())
print(f"target cycle {train.cycle_valid[itr[k]]}: soh_prev {ptr[k]:.4f} + dsoh {dtr[k]:+.4f} "
      f"= soh {train.soh[itr[k]]:.4f}")
w.round(4)

target cycle 11: soh_prev 0.9896 + dsoh -0.0059 = soh 0.9837


,cycle_valid,soh_prev,T_max_prev_C,chg_cc_time_s,Re_ohm,Rct_ohm,log_rest_h,is_4C,I_level_A,cutoff_V
0,2,1.0063,38.101799,3180.187988,0.0627,0.0916,0.4359,0.0,2.0,2.5
1,3,1.0000,38.149899,3178.187988,0.0614,0.0919,0.4360,0.0,2.0,2.5
2,4,0.9981,37.526699,3154.437012,0.0613,0.0914,0.4361,0.0,2.0,2.5
3,5,0.9934,37.062801,3129.266113,0.0614,0.0920,0.0178,0.0,2.0,2.5
4,6,0.9946,38.210999,3178.093994,0.0614,0.0920,0.0179,0.0,2.0,2.5
5,7,0.9923,38.174999,3163.797119,0.0614,0.0920,0.0180,0.0,2.0,2.5
6,8,0.9854,37.640400,3130.327881,0.0614,0.0920,0.0180,0.0,2.0,2.5
7,9,0.9822,37.375999,3111.422119,0.0625,0.0878,0.4365,0.0,2.0,2.5
8,10,0.9794,37.289299,3044.812988,0.0591,0.0899,0.0182,0.0,2.0,2.5
9,11,0.9896,37.523800,3143.672119,0.0591,0.0899,0.0182,0.0,2.0,2.5


ΔSOH per cycle is small and noisy compared with SOH itself; this is what the network has to learn beyond the naive forecast (ΔSOH = 0).

In [5]:
pd.Series(dtr).describe().round(4)

count    831.0000
mean      -0.0019
std        0.0144
min       -0.0554
25%       -0.0074
50%       -0.0034
75%        0.0002
max        0.1249
dtype: float64